# EDA — InvestLink Data Science Pipeline

**Autor:** InvestLink Data Science Team  
**Dataset:** `data/training_dataset.parquet` (gerado por `build_training_dataset.py`)  
**Objetivo:** Validar hipoteses de value investing no mercado brasileiro, entender a qualidade dos dados e guiar as decisoes de modelagem do Sprint 3.

---

## Hipoteses de Value Investing

| Indicador | BARATA (esperado) | CARA (esperado) | Logica |
|-----------|-------------------|-----------------|--------|
| P/L | Baixo (< 10) | Alto (> 20) | Preco baixo relativo ao lucro |
| P/VP | Baixo (< 1.5) | Alto (> 3) | Preco proximo ao valor patrimonial |
| ROE | Alto (> 15%) | Variavel | Alta rentabilidade sobre patrimonio |
| ROIC | Alto (> 12%) | Baixo | Retorno sobre capital investido |
| DY | Alto (> 5%) | Baixo | Dividendos como proxy de valuation |
| Div/EBIT | Baixo (< 2x) | Alto | Baixo endividamento = menos risco |
| Margem Liq. | Alta | Baixa ou negativa | Eficiencia operacional |
| EV/EBIT | Baixo (< 8x) | Alto (> 15x) | Valuation da empresa toda |
| PEG Ratio | Baixo (< 1) | Alto | P/L justificado pelo crescimento |
| Graham | Desconto (< 0%) | Premio (> 0%) | Margem de seguranca de Graham |

---

## Nota sobre Setores Brasileiros

O mercado brasileiro (IBXX) tem concentracao em **bancos, energia eletrica, commodities e varejo**.  
Isso significa que comparar P/L=15 entre um banco e uma varejista e enganoso — a normalizacao setorial e **indispensavel**.  
Setores com poucos tickers (< 5 empresas) podem ter z-scores ruidosos por falta de referencia intra-setor.

---

## Pre-requisitos

```bash
# 1. Criar tabelas no banco
python database/migrations.py

# 2. Scraping de indicadores e precos historicos
python web_scraping/run_scraping.py --mode all

# 3. Gerar dataset de treino com labels
python data_processing/build_training_dataset.py

# O arquivo ../data/training_dataset.parquet deve existir antes de rodar este notebook.
```

In [ ]:
import sys
import os
import warnings

sys.path.insert(0, os.path.dirname(os.getcwd()))
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from scipy import stats

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 100

DATASET_PATH = '../data/training_dataset.parquet'

LABEL_COLORS = {
    'BARATA': '#2ecc71',
    'NEUTRA': '#3498db',
    'CARA':   '#e74c3c'
}

LABEL_ORDER = ['BARATA', 'NEUTRA', 'CARA']

INDICATOR_COLS = [
    'dy', 'p_l', 'peg_ratio', 'p_vp', 'ev_ebitda', 'ev_ebit',
    'p_ebitda', 'p_ebit', 'vpa', 'p_ativo', 'lpa', 'p_sr',
    'p_cap_giro', 'p_ativo_circ_liq', 'div_liq_pl', 'div_liq_ebitda',
    'div_liq_ebit', 'pl_ativos', 'passivos_ativos', 'm_bruta',
    'm_ebitda', 'm_ebit', 'm_liquida', 'roe', 'roa', 'roic',
    'giro_ativos', 'liq_corrente', 'cagr_receitas_5', 'cagr_lucros_5',
    'graham_formula'
]

print('Configuracao carregada com sucesso.')
print(f'INDICATOR_COLS definidos: {len(INDICATOR_COLS)} colunas')

## 1. Carregamento e Visao Geral

O dataset e um **painel (panel data)**: cada linha representa um par `(ticker, year)`.  
Isso significa que a mesma empresa aparece multiplas vezes — uma por ano de cobertura.

**Estrutura esperada:**
- ~90 tickers do indice IBXX
- Anos: 2008 a 2023 (serie historica de 15 anos)
- ~40 colunas: indicadores brutos, z-scores setoriais, scores fatoriais, label
- Cada linha: snapshot anual dos fundamentos da empresa

**O que validar aqui:**
1. Shape razoavel (esperado ~1.000-1.500 linhas apos remocao de anos incompletos)
2. Cobertura uniforme de tickers por ano
3. Setores representativos do IBXX (banco, energia, commodities, varejo)
4. Labels distribuidos (NEUTRA deve dominar — e esperado)

In [ ]:
df = pd.read_parquet(DATASET_PATH)

print('=' * 55)
print('VISAO GERAL DO DATASET')
print('=' * 55)
print(f'Shape:          {df.shape[0]} linhas x {df.shape[1]} colunas')
print(f'Tickers unicos: {df["ticker"].nunique()}')
print(f'Periodo:        {df["year"].min()} a {df["year"].max()}')
print(f'Setores:        {df["sectorname"].nunique()}')
print()

label_counts = df['label'].value_counts()
print('Distribuicao de labels:')
for lbl in LABEL_ORDER:
    if lbl in label_counts:
        n = label_counts[lbl]
        pct = n / len(df) * 100
        print(f'  {lbl:8s}: {n:5d} ({pct:.1f}%)')

print()
print('Colunas do dataset:')
print(list(df.columns))

print()
print('Primeiras 3 linhas:')
df.head(3)

## 2. Analise de Valores Nulos

Mesmo apos o preenchimento com mediana do setor (`feature_engineer.py`), alguns nulos persistem.  
Entender **por que** eles existem e crucial para decidir o que fazer no treinamento.

### Por que nulos persistem apos imputacao?

| Indicador | Causa provavel de nulo |
|-----------|------------------------|
| `p_cap_giro` | Bancos nao tem capital de giro operacional — estruturalmente ausente |
| `peg_ratio` | Indefinido quando crescimento de lucros e negativo (divisao por negativo) |
| `graham_formula` | Indefinido quando LPA ou VPA sao negativos (raiz de negativo) |
| `p_ativo_circ_liq` | Negativo para empresas com passivo circulante > ativo circulante |
| `liq_corrente` | Holdings e financeiras nao reportam este indicador |
| `cagr_lucros_5` | Primeiros anos da serie (< 5 anos de historico) |
| `m_bruta` | Setores de intermediacao financeira nao tem margem bruta padrao |

**Setor pequeno:** se um setor tem < 3 empresas, a mediana do setor e pouco robusta e o preenchimento pode ser de baixa qualidade.

### Criterio de decisao para nulos

| % de nulos | Acao recomendada |
|------------|------------------|
| < 5%       | Imputar (mediana global) — impacto minimo |
| 5% - 20%   | Monitorar — pode usar indicador binario `_is_null` |
| > 20%      | Considerar remocao ou tratar como feature separada |

In [ ]:
existing_indicators = [c for c in INDICATOR_COLS if c in df.columns]
null_pct = df[existing_indicators].isna().mean().sort_values(ascending=False) * 100

def bar_color(pct):
    if pct > 20:
        return '#e74c3c'
    elif pct > 5:
        return '#f39c12'
    else:
        return '#2ecc71'

colors = [bar_color(v) for v in null_pct.values]

fig, ax = plt.subplots(figsize=(16, 5))
bars = ax.bar(range(len(null_pct)), null_pct.values, color=colors, alpha=0.85)
ax.set_xticks(range(len(null_pct)))
ax.set_xticklabels(null_pct.index, rotation=45, ha='right', fontsize=9)
ax.axhline(5,  color='#f39c12', linestyle='--', linewidth=1.5, label='5%  (monitorar)')
ax.axhline(20, color='#e74c3c', linestyle='--', linewidth=1.5, label='20% (remover/tratar)')
ax.set_title('Percentual de Nulos por Indicador (pos-imputacao de mediana setorial)', fontsize=13)
ax.set_ylabel('% de valores nulos')
ax.legend()
plt.tight_layout()
plt.show()

print('\nIndicadores CRITICOS (> 20% nulos):')
criticos = null_pct[null_pct > 20]
if len(criticos):
    for col, pct in criticos.items():
        print(f'  {col:25s}: {pct:.1f}%')
else:
    print('  Nenhum — bom sinal!')

print('\nIndicadores de ATENCAO (5-20% nulos):')
atencao = null_pct[(null_pct > 5) & (null_pct <= 20)]
if len(atencao):
    for col, pct in atencao.items():
        print(f'  {col:25s}: {pct:.1f}%')
else:
    print('  Nenhum')

## 3. Distribuicao dos Labels (Balanceamento de Classes)

As classes foram definidas com base no **alpha relativo ao Ibovespa** no ano seguinte:  
- **BARATA**: alpha > +15% (a acao superou o Ibov por mais de 15 p.p.)
- **CARA**: alpha < -15% (a acao ficou mais de 15 p.p. abaixo do Ibov)
- **NEUTRA**: o restante

**O que esperar:**  
NEUTRA deve dominar (~50-60% das observacoes), pois a maioria das acoes acompanha o mercado.  
BARATA e CARA devem ser mais raras e aproximadamente simetricas.

### Padroes historicos esperados por periodo

| Periodo | Esperado |
|---------|----------|
| 2008 (crise) | Mais CARA (queda generalizada) |
| 2009-2012 (recuperacao) | Mais BARATA (high-beta sobe mais) |
| 2014-2015 (recessao BR) | Mais CARA |
| 2016-2019 (reforma) | Misto, mais dispersao |
| 2020 (COVID) | Mais CARA no primeiro semestre |
| 2021 (recuperacao) | Mais BARATA (reversao setorial) |

### Impacto no modelo

Desequilibrio de classes nao e problema grave **se tratado corretamente**.  
A solucao adotada no Sprint 3 e `class_weight='balanced'` (equivalente a `sample_weight` por classe),  
combinado com metrica primaria **F1-weighted** em vez de acuracia.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Subplot 1: contagem absoluta
counts = df['label'].value_counts()
labels_ordered = [l for l in LABEL_ORDER if l in counts.index]
vals = [counts[l] for l in labels_ordered]
bar_colors = [LABEL_COLORS[l] for l in labels_ordered]
axes[0].bar(labels_ordered, vals, color=bar_colors, alpha=0.85, edgecolor='white')
axes[0].set_title('Contagem absoluta por label')
axes[0].set_ylabel('Qtd registros')
for i, v in enumerate(vals):
    axes[0].text(i, v + 5, str(v), ha='center', fontweight='bold')

# Subplot 2: stacked bar por ano
year_label = df.groupby(['year', 'label']).size().unstack(fill_value=0)
bottom = np.zeros(len(year_label))
years = year_label.index.tolist()
for lbl in LABEL_ORDER:
    if lbl in year_label.columns:
        axes[1].bar(years, year_label[lbl].values, bottom=bottom,
                    color=LABEL_COLORS[lbl], alpha=0.85, label=lbl, edgecolor='white')
        bottom += year_label[lbl].values
axes[1].set_title('Labels por ano (absoluto)')
axes[1].set_ylabel('Qtd registros')
axes[1].legend(title='Label')
axes[1].set_xticks(years)
axes[1].set_xticklabels(years, rotation=45)

# Subplot 3: area chart % por ano
year_label_pct = year_label.div(year_label.sum(axis=1), axis=0) * 100
bottom_pct = np.zeros(len(year_label_pct))
for lbl in LABEL_ORDER:
    if lbl in year_label_pct.columns:
        axes[2].fill_between(years, bottom_pct, bottom_pct + year_label_pct[lbl].values,
                             color=LABEL_COLORS[lbl], alpha=0.75, label=lbl)
        bottom_pct += year_label_pct[lbl].values
axes[2].set_title('Proporcao % por ano (area chart)')
axes[2].set_ylabel('%')
axes[2].set_ylim(0, 100)
axes[2].legend(title='Label')
axes[2].set_xticks(years)
axes[2].set_xticklabels(years, rotation=45)

plt.tight_layout()
plt.show()

# Imbalance ratio
n_majority = counts.max()
n_minority = counts.min()
ratio = n_majority / n_minority
print(f'\nImbalance ratio (majoritaria / minoritaria): {ratio:.2f}x')
if ratio < 2:
    print('Interpretacao: balanceamento razoavel — sample_weight pode ser suficiente.')
elif ratio < 4:
    print('Interpretacao: desequilibrio moderado — usar class_weight="balanced" obrigatoriamente.')
else:
    print('Interpretacao: desequilibrio severo — considerar SMOTE ou threshold tuning.')

## 4. Retornos e Limiares de Classificacao

### Por que +-15%?

O limiar de **+-15% de alpha** foi escolhido empiricamente para:
1. Garantir separacao **economicamente significativa** (nao apenas estatistica)
2. Excluir ruido — alphas de +-5% podem ser apenas volatilidade, nao sinal real
3. Manter classes minoritarias com amostras suficientes para treinar

### O que observar nesta secao

- **Histograma por label**: as distribuicoes devem ser claramente separadas
- **Boxplot**: BARATA deve ter mediana positiva, CARA negativa, NEUTRA proxima de zero
- **Q-Q plot**: caudas pesadas sao esperadas em retornos financeiros — nao assuma normalidade
- **Shapiro-Wilk**: confirmar que as distribuicoes NAO sao normais (financas nao sao)

Se as distribuicoes se sobreporem muito, o limiar pode estar muito conservador ou o alpha calculado tem ruido.

In [ ]:
if 'alpha' not in df.columns:
    print('Coluna alpha nao encontrada. Pulando secao 4.')
else:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Histograma por label
    for lbl in LABEL_ORDER:
        if lbl in df['label'].unique():
            subset = df[df['label'] == lbl]['alpha'].dropna()
            axes[0].hist(subset, bins=40, alpha=0.5,
                         color=LABEL_COLORS[lbl], label=f'{lbl} (n={len(subset)})', density=True)
    axes[0].axvline(0.15,  color='#27ae60', linestyle='--', linewidth=2, label='+15% (limiar)')
    axes[0].axvline(-0.15, color='#c0392b', linestyle='--', linewidth=2, label='-15% (limiar)')
    axes[0].axvline(0,     color='black',   linestyle=':',  linewidth=1, label='Zero')
    axes[0].set_title('Distribuicao do alpha por label')
    axes[0].set_xlabel('Alpha (retorno vs Ibovespa)')
    axes[0].xaxis.set_major_formatter(mtick.PercentFormatter(1.0))
    axes[0].legend(fontsize=8)

    # Boxplot por label
    data_box = [df[df['label'] == lbl]['alpha'].dropna().values for lbl in LABEL_ORDER if lbl in df['label'].unique()]
    bp = axes[1].boxplot(data_box, labels=LABEL_ORDER, patch_artist=True, showfliers=False)
    for patch, lbl in zip(bp['boxes'], LABEL_ORDER):
        patch.set_facecolor(LABEL_COLORS[lbl])
        patch.set_alpha(0.7)
    for median in bp['medians']:
        median.set_color('black')
        median.set_linewidth(2)
    axes[1].axhline(0, color='gray', linestyle=':', linewidth=1)
    axes[1].set_title('Boxplot do alpha por label')
    axes[1].set_ylabel('Alpha')
    axes[1].yaxis.set_major_formatter(mtick.PercentFormatter(1.0))

    # Q-Q plot do alpha total
    alpha_vals = df['alpha'].dropna().values
    (osm, osr), (slope, intercept, r) = stats.probplot(alpha_vals, dist='norm')
    axes[2].scatter(osm, osr, alpha=0.3, s=10, color='steelblue', label='Dados')
    line_x = np.array([osm.min(), osm.max()])
    axes[2].plot(line_x, slope * line_x + intercept, color='red', linewidth=2, label='Normal teorica')
    axes[2].set_title('Q-Q Plot do alpha (vs Normal)')
    axes[2].set_xlabel('Quantis teoricos (Normal)')
    axes[2].set_ylabel('Quantis observados')
    axes[2].legend()

    plt.suptitle('Analise dos retornos (alpha vs Ibovespa)', fontsize=13, y=1.01)
    plt.tight_layout()
    plt.show()

    print('\nEstatisticas do alpha por label:')
    stats_df = df.groupby('label')['alpha'].agg(['mean','median','std',
        lambda x: stats.skew(x.dropna()),
        lambda x: stats.kurtosis(x.dropna())
    ])
    stats_df.columns = ['mean','median','std','skew','kurtosis']
    print(stats_df.round(3).to_string())

    print('\nTeste Shapiro-Wilk por label (H0: dados sao normais):')
    for lbl in LABEL_ORDER:
        subset = df[df['label'] == lbl]['alpha'].dropna()
        if len(subset) >= 3:
            sample = subset.sample(min(500, len(subset)), random_state=42)
            stat_sw, p_sw = stats.shapiro(sample)
            resultado = 'REJEITA H0 (nao normal)' if p_sw < 0.05 else 'Nao rejeita H0 (normal)'
            print(f'  {lbl:8s}: p={p_sw:.4f} -> {resultado}')

In [ ]:
if 'alpha' in df.columns:
    alpha_all = df['alpha'].dropna()

    print('Percentis do alpha (todos os labels):')
    percentiles = [25, 33, 50, 67, 75]
    for p in percentiles:
        print(f'  P{p:2d}: {np.percentile(alpha_all, p)*100:.1f}%')

    print('\nSensibilidade do limiar:')
    print(f'{"Limiar":>8} | {"% BARATA":>9} | {"% NEUTRA":>9} | {"% CARA":>9} | {"Total rows":>10}')
    print('-' * 55)
    for limiar in [0.10, 0.12, 0.15, 0.18, 0.20, 0.25]:
        barata = (alpha_all > limiar).mean() * 100
        cara   = (alpha_all < -limiar).mean() * 100
        neutra = 100 - barata - cara
        total  = len(alpha_all)
        print(f'  +-{limiar*100:.0f}%   | {barata:8.1f}% | {neutra:8.1f}% | {cara:8.1f}% | {total:10d}')

## 5. Indicadores por Label — Validando a Hipotese de Value Investing

### O que esperar para cada indicador

| Indicador | Esperado em BARATA | Esperado em CARA | Logica |
|-----------|-------------------|------------------|--------|
| P/L | Histograma deslocado para esquerda | Deslocado para direita | Preco mais baixo por unidade de lucro |
| P/VP | Pico abaixo de 2 | Pico acima de 3 | Desconto ao valor patrimonial |
| ROE | Media mais alta | Media mais baixa ou variavel | Alta rentabilidade atrai preco, mas pode ser leading indicator |
| ROIC | Mais alto | Mais baixo | Capital bem alocado sustenta vantagem competitiva |
| Margem Liq. | Mais alta | Mais baixa | Eficiencia converte receita em lucro |
| DY | Mais alto | Mais baixo | Dividendo alto com preco baixo = yield alto |
| Div/EBIT | Mais baixo | Mais alto | Empresa menos endividada = menos risco financeiro |
| EV/EBIT | Mais baixo | Mais alto | Valuation total da empresa |

**Nota sobre z-scores:** quando os indicadores brutos nao separam as classes bem,  
os z-scores setoriais (que compensam diferencas entre setores) tendem a separar melhor.  
Isso e evidencia de que a normalizacao setorial adiciona valor preditivo.

In [ ]:
key_indicators = ['p_l', 'p_vp', 'roe', 'roic', 'm_liquida', 'dy', 'div_liq_ebit', 'ev_ebit']
key_indicators = [c for c in key_indicators if c in df.columns]

n_cols = 4
n_rows = 2
fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 9))
axes = axes.flatten()

for i, col in enumerate(key_indicators):
    for lbl in LABEL_ORDER:
        if lbl in df['label'].unique():
            subset = df[df['label'] == lbl][col].dropna()
            # Clamp extremos para visualizacao mais limpa
            p1, p99 = subset.quantile(0.01), subset.quantile(0.99)
            subset_clamped = subset.clip(p1, p99)
            axes[i].hist(subset_clamped, bins=30, alpha=0.5,
                         color=LABEL_COLORS[lbl], label=lbl, density=True)
    axes[i].set_title(col, fontsize=11, fontweight='bold')
    axes[i].legend(fontsize=7)
    axes[i].set_xlabel('Valor (P1-P99)')

# Ocultar axes extras
for j in range(len(key_indicators), len(axes)):
    axes[j].set_visible(False)

plt.suptitle(
    'Distribuicao dos principais indicadores por label\n'
    '(Hipotese de value investing: BARATA deve ter P/L e P/VP menores, ROE e DY maiores)',
    fontsize=12, y=1.01
)
plt.tight_layout()
plt.show()

In [ ]:
from scipy.stats import kruskal

print('Teste Kruskal-Wallis por indicador (H0: distribuicoes iguais entre labels)')
print(f'{"Indicador":25} | {"H-stat":>8} | {"p-valor":>10} | Significancia')
print('-' * 65)

for col in existing_indicators:
    if col not in df.columns:
        continue
    groups = [df[df['label'] == lbl][col].dropna().values for lbl in LABEL_ORDER if lbl in df['label'].unique()]
    groups = [g for g in groups if len(g) >= 3]
    if len(groups) < 2:
        continue
    try:
        h_stat, p_val = kruskal(*groups)
        if p_val < 0.001:
            sig = 'Significativo ***'
        elif p_val < 0.01:
            sig = 'Significativo **'
        elif p_val < 0.05:
            sig = 'Significativo *'
        elif p_val < 0.10:
            sig = 'Marginal .'
        else:
            sig = 'Nao significativo'
        print(f'{col:25} | {h_stat:8.2f} | {p_val:10.4f} | {sig}')
    except Exception:
        pass

### Interpretacao do Teste Kruskal-Wallis

| Resultado | Significado para o modelo |
|-----------|---------------------------|
| Significativo *** (p < 0.001) | Indicador tem forte poder discriminativo entre labels. Manter com prioridade alta. |
| Significativo * (p < 0.05) | Poder discriminativo moderado. Manter, mas nao depender sozinho. |
| Marginal (p < 0.10) | Separacao fraca — util em conjunto, nao como feature principal. |
| Nao significativo (p >= 0.10) | Distribuicoes similares entre labels. Considerar remocao ou combinacao com outros. |

**Recomendacao para Sprint 3:** priorizar features com p < 0.01 e z-scores correspondentes.  
Features nao significativas podem ser mantidas se o Random Forest as considera importantes — o modelo pode capturar interacoes que o teste univariado nao ve.

## 6. Correlacao entre Indicadores

### Por que multicolinearidade importa?

**Para modelos lineares:**
- Instabilidade numerica nos coeficientes
- Dificuldade em interpretar a importancia de cada feature
- Overfitting mascarado por R2 alto

**Para GradientBoosting/XGBoost:** o impacto e menor, mas ainda existe:
- Features altamente correlacionadas competem entre si, dividindo importancia
- Dataset maior que necessario — custo computacional sem ganho preditivo
- Interpretabilidade comprometida (qual das duas features correlacionadas e a 'real'?)

### Correlacoes altas esperadas

| Grupo | Features correlacionadas | Motivo |
|-------|--------------------------|--------|
| Cascade de margem | m_bruta, m_ebitda, m_ebit, m_liquida | Cada uma e subset da anterior |
| Valuation | p_l, p_ebitda, p_ebit, ev_ebitda, ev_ebit | Multiplos do mesmo denominador |
| Qualidade | roe, roic, roa | Retorno sobre diferentes bases de capital |
| Endividamento | div_liq_pl, div_liq_ebitda, div_liq_ebit | Divida sobre diferentes denominadores |

### Thresholds de VIF (Variance Inflation Factor)

| VIF | Interpretacao |
|-----|---------------|
| < 5 | Aceitavel |
| 5-10 | Atencao — considerar remocao |
| > 10 | Problematico — remover uma das features do grupo |

In [ ]:
corr_cols = [c for c in existing_indicators if c in df.columns]
corr = df[corr_cols].corr()

mask = np.triu(np.ones_like(corr, dtype=bool))

fig, ax = plt.subplots(figsize=(16, 13))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f',
    cmap='RdBu_r', center=0, vmin=-1, vmax=1,
    linewidths=0.4, annot_kws={'size': 7}, ax=ax,
    cbar_kws={'shrink': 0.8}
)
ax.set_title('Matriz de Correlacao — Indicadores Financeiros\n(triangulo inferior | vermelho=positivo, azul=negativo)',
             fontsize=12, pad=15)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(fontsize=8)
plt.tight_layout()
plt.show()

# Pares com alta correlacao (|r| > 0.7)
print('\nPares com correlacao alta (|r| > 0.7):')
high_corr = []
for i, c1 in enumerate(corr_cols):
    for j, c2 in enumerate(corr_cols):
        if i < j and abs(corr.loc[c1, c2]) > 0.7:
            high_corr.append((c1, c2, corr.loc[c1, c2]))
high_corr.sort(key=lambda x: abs(x[2]), reverse=True)
for c1, c2, r in high_corr:
    print(f'  {c1:20} x {c2:20}: r={r:.3f}')

In [ ]:
try:
    from statsmodels.stats.outliers_influence import variance_inflation_factor
    from statsmodels.tools.tools import add_constant

    vif_cols = [c for c in corr_cols if c in df.columns]
    df_vif = df[vif_cols].dropna()

    # Limitar a colunas com variancia > 0
    vif_cols_valid = [c for c in vif_cols if df_vif[c].std() > 0]
    df_vif = df_vif[vif_cols_valid]

    X_vif = add_constant(df_vif)
    vif_data = pd.DataFrame({
        'feature': vif_cols_valid,
        'VIF': [variance_inflation_factor(X_vif.values, i + 1) for i in range(len(vif_cols_valid))]
    }).sort_values('VIF', ascending=False)

    print('VIF por indicador (ordenado, maior = mais multicolinear):')
    print(f'{"Feature":25} | {"VIF":>8} | Status')
    print('-' * 45)
    for _, row in vif_data.iterrows():
        vif_val = row['VIF']
        if vif_val > 10:
            status = 'PROBLEMATICO'
        elif vif_val > 5:
            status = 'Atencao'
        else:
            status = 'OK'
        print(f'{row["feature"]:25} | {vif_val:8.2f} | {status}')

except ImportError:
    print('statsmodels nao instalado. Instale com: pip install statsmodels')
    print('Alternativa: usar correlacao > 0.9 como proxy de multicolinearidade severa.')
    very_high = [(c1, c2, r) for c1, c2, r in high_corr if abs(r) > 0.9]
    if very_high:
        print('\nPares com |r| > 0.9 (provavel VIF > 10):')
        for c1, c2, r in very_high:
            print(f'  {c1} x {c2}: r={r:.3f}')

### Estrategia de tratamento de multicolinearidade

**Opcao 1 — Remover o menos importante:** entre duas features altamente correlacionadas, manter a que tem maior importancia no Random Forest.  
**Opcao 2 — Combinar:** criar um score composto (ja feito para margens, valuation, qualidade e crescimento).  
**Opcao 3 — Nao fazer nada:** GradientBoosting e XGBoost sao robustos a multicolinearidade — o impacto e principalmente na interpretabilidade.

**Recomendacao concreta:**
- Do grupo de margens (`m_bruta`, `m_ebitda`, `m_ebit`, `m_liquida`): manter apenas `m_liquida` e `m_ebit`
- Do grupo de retorno sobre capital (`roe`, `roa`, `roic`): manter `roic` como primario
- Do grupo de divida (`div_liq_pl`, `div_liq_ebitda`, `div_liq_ebit`): manter `div_liq_ebit` (mais comum em analise de credito)

## 7. Analise por Setor

### Por que a normalizacao setorial e indispensavel?

Um P/L de 15x pode ser **caro** para um banco (tipicamente 8-12x),  
**normal** para uma empresa de energia (12-18x),  
e **barato** para uma empresa de tecnologia (normalmente > 30x).

Comparar indicadores sem ajuste setorial e como comparar velocidades de carro e aviao.  
O z-score setorial resolve isso: transforma cada indicador em 'quantos desvios padrao acima/abaixo da mediana do setor'.

**O que observar aqui:**
- Setores com muito poucos registros (< 30) tem z-scores ruidosos
- P/L e ROE medios variam muito entre setores — confirmando a necessidade de normalizacao
- A distribuicao de labels por setor revela setores sistematicamente sub ou sobreavaliados

In [ ]:
sector_counts_all = df.groupby('sectorname').size().sort_values(ascending=False)

def sector_bar_color(n):
    if n < 30:
        return '#e74c3c'
    elif n < 60:
        return '#f39c12'
    else:
        return '#2ecc71'

sec_colors = [sector_bar_color(n) for n in sector_counts_all.values]

fig, ax = plt.subplots(figsize=(16, 5))
ax.bar(range(len(sector_counts_all)), sector_counts_all.values, color=sec_colors, alpha=0.85, edgecolor='white')
ax.set_xticks(range(len(sector_counts_all)))
ax.set_xticklabels(sector_counts_all.index, rotation=45, ha='right', fontsize=9)
ax.set_title('Registros por Setor (verde > 60, amarelo 30-60, vermelho < 30)', fontsize=12)
ax.set_ylabel('Qtd registros (linhas no dataset)')

# Legenda manual
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#2ecc71', label='> 60 registros (robusto)'),
    Patch(facecolor='#f39c12', label='30-60 registros (atencao)'),
    Patch(facecolor='#e74c3c', label='< 30 registros (z-score ruidoso)')
]
ax.legend(handles=legend_elements, loc='upper right')
plt.tight_layout()
plt.show()

# Medianas por setor ordenadas por ROIC
cols_setor = [c for c in ['p_l', 'roe', 'roic', 'dy', 'm_liquida'] if c in df.columns]
if cols_setor:
    sector_medians = df.groupby('sectorname')[cols_setor].median().sort_values('roic' if 'roic' in cols_setor else cols_setor[0], ascending=False)
    print('\nMedianas por setor (ordenado por ROIC decrescente):')
    print(sector_medians.round(2).to_string())

In [ ]:
# P/L e ROE por setor -- evidencia da necessidade de normalizacao
sectors_with_data = sector_counts_all[sector_counts_all >= 10].index.tolist()
df_sectors = df[df['sectorname'].isin(sectors_with_data)]

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

for ax, col, title in [
    (axes[0], 'p_l',  'P/L por Setor'),
    (axes[1], 'roe',  'ROE por Setor')
]:
    if col not in df.columns:
        ax.text(0.5, 0.5, f'{col} nao disponivel', transform=ax.transAxes, ha='center')
        continue

    sector_order = df_sectors.groupby('sectorname')[col].median().sort_values().index.tolist()
    data_by_sector = [df_sectors[df_sectors['sectorname'] == s][col].dropna().clip(
        df_sectors[col].quantile(0.01), df_sectors[col].quantile(0.99)
    ).values for s in sector_order]

    bp = ax.boxplot(data_by_sector, labels=sector_order, patch_artist=True,
                    showfliers=False, vert=True)
    for patch in bp['boxes']:
        patch.set_facecolor('steelblue')
        patch.set_alpha(0.6)
    for median in bp['medians']:
        median.set_color('red')
        median.set_linewidth(2)
    ax.set_title(f'{title}\n(evidencia: setores tem distribuicoes muito diferentes)', fontsize=10)
    ax.set_xticklabels(sector_order, rotation=45, ha='right', fontsize=8)

plt.suptitle('P/L e ROE por setor — por que a normalizacao setorial e essencial', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Labels por setor -- stacked bar
sector_label = df.groupby(['sectorname', 'label']).size().unstack(fill_value=0)
sector_label_pct = sector_label.div(sector_label.sum(axis=1), axis=0) * 100

# Ordenar por % BARATA
if 'BARATA' in sector_label_pct.columns:
    sector_label_pct = sector_label_pct.sort_values('BARATA', ascending=True)

fig, ax = plt.subplots(figsize=(16, 6))
bottom_arr = np.zeros(len(sector_label_pct))
for lbl in LABEL_ORDER:
    if lbl in sector_label_pct.columns:
        ax.barh(range(len(sector_label_pct)), sector_label_pct[lbl].values,
                left=bottom_arr, color=LABEL_COLORS[lbl], alpha=0.85, label=lbl)
        bottom_arr += sector_label_pct[lbl].values

ax.set_yticks(range(len(sector_label_pct)))
ax.set_yticklabels(sector_label_pct.index, fontsize=9)
ax.set_xlabel('% de registros')
ax.set_title('Distribuicao de labels por setor (ordenado por % BARATA)', fontsize=12)
ax.legend(title='Label', bbox_to_anchor=(1.02, 1), loc='upper left')
ax.axvline(50, color='gray', linestyle='--', linewidth=1)
plt.tight_layout()
plt.show()

if 'BARATA' in sector_label_pct.columns:
    top_barata = sector_label_pct['BARATA'].sort_values(ascending=False).head(5)
    print('\nSetores com maior % BARATA historicamente:')
    for setor, pct in top_barata.items():
        print(f'  {setor:30}: {pct:.1f}%')

### Interpretacao dos Resultados Setoriais

**Value traps vs oportunidades estruturais:**  
Um setor com % BARATA historicamente alta pode indicar duas coisas:
1. **Oportunidade estrutural:** setor genuinamente subavaliado pelo mercado (ex: utilities, bancos em crises)
2. **Value trap:** setor com problemas estruturais que justificam valuation baixo (ex: varejistas tradicionais com disrupcao digital)

**Nao usar nome do setor como feature diretamente:**  
O setor e uma variavel categorica de alta cardinalidade — codifica-la via one-hot cria dimensionalidade desnecessaria.  
A normalizacao setorial (z-scores) ja captura a informacao setorial de forma mais eficiente.

## 8. Composite Scores — Validacao dos Scores Fatoriais

### Logica dos 4 fatores

O composite score foi construido como media ponderada de 4 fatores:

| Fator | Peso | Indicadores base | Justificativa |
|-------|------|------------------|---------------|
| Value | 30% | P/L, P/VP, EV/EBIT, EV/EBITDA | Valuation absoluto e relativo |
| Quality | 35% | ROE, ROIC, Margem Liquida, Div/EBIT | Qualidade do negocio e sustentabilidade |
| Growth | 20% | CAGR Receitas 5a, CAGR Lucros 5a | Crescimento historico como proxy de momentum |
| Dividend | 15% | DY, Payout | Retorno direto ao acionista |

**Por que Quality tem peso maior?**  
Empresas de alta qualidade (alto ROE, baixo endividamento) tendem a ser mais resilientes e a gerar alpha de longo prazo com mais consistencia do que empresas baratas de baixa qualidade.

### O que validar
- **BARATA** deve ter composite_score mais alto (e Value_score e Quality_score especialmente)
- **CARA** pode ter Growth_score alto — isso e esperado e correto (mercado paga premio por crescimento)
- **Histograma:** distribuicoes dos scores devem separar as classes
- **ANOVA:** testar se as medias dos scores diferem significativamente entre labels

In [ ]:
score_cols = [c for c in ['value_score', 'quality_score', 'growth_score', 'dividend_score', 'composite_score']
              if c in df.columns]

if not score_cols:
    print('Colunas de score nao encontradas no dataset.')
else:
    fig, axes = plt.subplots(1, len(score_cols), figsize=(4 * len(score_cols), 5))
    if len(score_cols) == 1:
        axes = [axes]

    for ax, col in zip(axes, score_cols):
        for lbl in LABEL_ORDER:
            if lbl in df['label'].unique():
                subset = df[df['label'] == lbl][col].dropna()
                ax.hist(subset, bins=20, alpha=0.55,
                        color=LABEL_COLORS[lbl], label=f'{lbl} (md={subset.median():.0f})', density=True)
        ax.set_title(col.replace('_', ' ').title(), fontsize=10, fontweight='bold')
        ax.set_xlabel('Score (0-100)')
        ax.legend(fontsize=7)

    plt.suptitle('Distribuicao dos Scores Fatoriais por Label\n(BARATA deve liderar Value e Quality)', fontsize=12, y=1.01)
    plt.tight_layout()
    plt.show()

    # ANOVA para cada score
    print('\nANOVA one-way por score (H0: medias iguais entre labels):')
    print(f'{"Score":25} | {"F-stat":>8} | {"p-valor":>10} | Interpretacao')
    print('-' * 65)
    for col in score_cols:
        groups = [df[df['label'] == lbl][col].dropna().values
                  for lbl in LABEL_ORDER if lbl in df['label'].unique()]
        groups = [g for g in groups if len(g) >= 3]
        if len(groups) >= 2:
            f_stat, p_val = stats.f_oneway(*groups)
            interp = 'Scores diferem significativamente' if p_val < 0.05 else 'Sem diferenca significativa'
            print(f'{col:25} | {f_stat:8.2f} | {p_val:10.4f} | {interp}')

In [ ]:
if score_cols and len(score_cols) >= 3:
    factor_scores = [c for c in ['value_score', 'quality_score', 'growth_score', 'dividend_score']
                     if c in score_cols]

    if factor_scores:
        # Perfil medio por label para radar chart
        profile = df.groupby('label')[factor_scores].mean()

        N = len(factor_scores)
        angles = [n / float(N) * 2 * np.pi for n in range(N)]
        angles += angles[:1]  # fechar o poligono

        fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

        for lbl in LABEL_ORDER:
            if lbl not in profile.index:
                continue
            values = profile.loc[lbl, factor_scores].tolist()
            values += values[:1]
            ax.plot(angles, values, linewidth=2, linestyle='solid',
                    color=LABEL_COLORS[lbl], label=lbl)
            ax.fill(angles, values, color=LABEL_COLORS[lbl], alpha=0.15)

        ax.set_xticks(angles[:-1])
        ax.set_xticklabels([c.replace('_score', '').title() for c in factor_scores], fontsize=11)
        ax.set_ylim(0, 100)
        ax.set_title('Perfil Medio dos Scores Fatoriais por Label\n(radar chart)', fontsize=12, pad=20)
        ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
        plt.tight_layout()
        plt.show()

        print('\nMedia dos scores por label:')
        print(profile.round(1).to_string())

### Interpretacao do Radar Chart

**Se CARA tem Growth_score maior que BARATA:** isso e **esperado e correto**.  
O mercado paga multiplos maiores por empresas de crescimento — isso e o momentum premium.  
O modelo deve aprender que alto crescimento COM alto valuation = CARA (crescimento ja precificado).

**Se BARATA nao lidera Value_score:** isso seria um sinal de alerta.  
Pode indicar que o limiar de +15% de alpha captura empresas que subiram por outros fatores (especulacao, baixa liquidez), nao por fundamentalismo.

## 9. Formula de Graham — Margem de Seguranca

### A formula

Graham propoe que o valor intrinseco de uma acao e aproximado por:

```
Valor Graham = sqrt(22.5 * LPA * VPA)
```

Onde:
- **LPA** = Lucro por Acao
- **VPA** = Valor Patrimonial por Acao
- **22.5** = 15x PE * 1.5x PB (o maximo que Graham aceitaria pagar)

**Interpretacao:**
- Desconto positivo (preco < Graham): acao barata com margem de seguranca
- Desconto negativo (preco > Graham): acao cara relativo ao valor intrinseco

**O % com NaN e informativo:**  
Empresas com LPA ou VPA negativo nao tem formula valida (raiz de numero negativo).  
Isso significa que empresas com prejuizo ou patrimonio negativo sao excluidas da analise de Graham — e uma limitacao conhecida e intencional do metodo.

**Conservadorismo:** Graham foi desenvolvido para mercados desenvolvidos. No Brasil, as taxas de juros mais altas implicam custo de oportunidade maior, tornando o criterio ainda mais rigoroso.

In [ ]:
if 'graham_formula' not in df.columns:
    print('Coluna graham_formula nao encontrada.')
else:
    # Calcular desconto ao Graham (requer preco atual)
    price_col = None
    for candidate in ['price_current', 'preco_atual', 'price', 'cotacao']:
        if candidate in df.columns:
            price_col = candidate
            break

    graham_valid = df[df['graham_formula'] > 0].copy()
    n_total = len(df)
    n_valid = len(graham_valid)
    n_null = df['graham_formula'].isna().sum()
    n_negative = (df['graham_formula'] <= 0).sum() - n_null

    print(f'Registros com Graham valido: {n_valid}/{n_total} ({n_valid/n_total*100:.1f}%)')
    print(f'Excluidos por NaN (LPA ou VPA nulos): {n_null}')
    print(f'Excluidos por valor negativo (LPA ou VPA negativos): {n_negative}')

    if price_col:
        graham_valid['discount_pct'] = (
            (graham_valid['graham_formula'] - graham_valid[price_col]) / graham_valid['graham_formula'] * 100
        )

        fig, axes = plt.subplots(1, 3, figsize=(18, 5))

        # Histograma do desconto por label
        for lbl in LABEL_ORDER:
            if lbl in graham_valid['label'].unique():
                subset = graham_valid[graham_valid['label'] == lbl]['discount_pct'].dropna()
                subset = subset.clip(-200, 200)
                axes[0].hist(subset, bins=30, alpha=0.5,
                             color=LABEL_COLORS[lbl], label=lbl, density=True)
        axes[0].axvline(0, color='black', linestyle='--', linewidth=2, label='Preco = Graham')
        axes[0].set_title('Desconto ao Graham por label')
        axes[0].set_xlabel('Desconto (%) — positivo = barato vs Graham')
        axes[0].legend(fontsize=8)

        # Boxplot
        data_box = [graham_valid[graham_valid['label'] == lbl]['discount_pct'].dropna().clip(-200, 200).values
                    for lbl in LABEL_ORDER if lbl in graham_valid['label'].unique()]
        bp = axes[1].boxplot(data_box, labels=LABEL_ORDER, patch_artist=True, showfliers=False)
        for patch, lbl in zip(bp['boxes'], LABEL_ORDER):
            patch.set_facecolor(LABEL_COLORS[lbl])
            patch.set_alpha(0.7)
        for median in bp['medians']:
            median.set_color('black')
            median.set_linewidth(2)
        axes[1].axhline(0, color='gray', linestyle=':', linewidth=1)
        axes[1].set_title('Boxplot desconto ao Graham')
        axes[1].set_ylabel('Desconto (%)')

        # % abaixo do Graham por label
        below_graham = {}
        for lbl in LABEL_ORDER:
            if lbl in graham_valid['label'].unique():
                subset = graham_valid[graham_valid['label'] == lbl]['discount_pct'].dropna()
                below_graham[lbl] = (subset > 0).mean() * 100

        axes[2].bar(list(below_graham.keys()), list(below_graham.values()),
                    color=[LABEL_COLORS[l] for l in below_graham.keys()], alpha=0.85, edgecolor='white')
        axes[2].set_title('% de acoes abaixo do valor de Graham por label')
        axes[2].set_ylabel('% de registros')
        axes[2].axhline(50, color='gray', linestyle='--', linewidth=1, label='50%')
        axes[2].legend()
        for i, (lbl, pct) in enumerate(below_graham.items()):
            axes[2].text(i, pct + 1, f'{pct:.1f}%', ha='center', fontweight='bold')

        plt.suptitle('Analise da Formula de Graham — Margem de Seguranca', fontsize=12, y=1.01)
        plt.tight_layout()
        plt.show()
    else:
        print('Coluna de preco atual nao encontrada. Mostrando apenas graham_formula por label:')
        print(graham_valid.groupby('label')['graham_formula'].describe().round(2).to_string())

## 10. Z-scores Setoriais — Validacao da Normalizacao

### Propriedades esperadas dos z-scores

Apos normalizacao setorial (z = (x - media_setor) / std_setor):
- **Media ~0**: cada setor contribui simetricamente
- **Valores entre -3 e +3**: outliers extremos sao raros
- **Distribuicao mais simetrica** que os indicadores brutos
- **Curva Normal teorica** serve como referencia — pequenos desvios sao normais

### Por que pode haver desvio de 0

| Causa | O que fazer |
|-------|-------------|
| Setores com < 3 empresas | Z-score nao e confiavel — considerar excluir esses setores |
| Distribuicoes assimetricas no setor | Normal. Medianas de setores com outliers puxam a media do z-score |
| Outliers nao removidos | Winsorizar antes de calcular z-score |

**Alerta:** se media do z-score > |0.3|, indica problema na implementacao da normalizacao.  
A curva Normal em vermelho serve como referencia visual — nao como requisito estatistico.

In [ ]:
z_cols = sorted([c for c in df.columns if c.endswith('_z')])

if not z_cols:
    print('Nenhuma coluna _z encontrada. Pulando secao 10.')
else:
    n_z = len(z_cols)
    n_cols_grid = 4
    n_rows_grid = int(np.ceil(n_z / n_cols_grid))

    fig, axes = plt.subplots(n_rows_grid, n_cols_grid, figsize=(5 * n_cols_grid, 4 * n_rows_grid))
    axes_flat = axes.flatten() if hasattr(axes, 'flatten') else [axes]

    x_range = np.linspace(-4, 4, 200)
    normal_pdf = stats.norm.pdf(x_range, 0, 1)

    for i, col in enumerate(z_cols):
        ax = axes_flat[i]
        data = df[col].dropna().clip(-4, 4)
        ax.hist(data, bins=35, alpha=0.65, color='steelblue', density=True, label='Dados')
        ax.plot(x_range, normal_pdf, color='red', linewidth=1.5, linestyle='--', label='Normal(0,1)')
        ax.axvline(data.mean(), color='orange', linewidth=1.5, linestyle='-',
                   label=f'Media={data.mean():.2f}')
        ax.set_title(col, fontsize=9, fontweight='bold')
        ax.set_xlim(-4, 4)
        ax.legend(fontsize=6)

    for j in range(n_z, len(axes_flat)):
        axes_flat[j].set_visible(False)

    plt.suptitle('Z-scores Setoriais — Distribuicao vs Normal Teorica (linha vermelha)',
                 fontsize=12, y=1.01)
    plt.tight_layout()
    plt.show()

    # Tabela de estatisticas
    z_stats = df[z_cols].agg(['mean', 'std', 'skew']).T
    z_stats.columns = ['Media', 'Std', 'Skew']
    z_stats = z_stats.round(3)
    print('\nEstatisticas dos z-scores (Media ideal ~0, Std ideal ~1):')
    print(z_stats.to_string())

    # Alertas
    problemas = z_stats[z_stats['Media'].abs() > 0.3]
    if len(problemas):
        print(f'\nALERTA: {len(problemas)} z-score(s) com media fora de [-0.3, +0.3]:')
        for col in problemas.index:
            print(f'  {col}: media={z_stats.loc[col, "Media"]:.3f}')
        print('  -> Verificar implementacao da normalizacao no feature_engineer.py')
    else:
        print('\nTodos os z-scores com media dentro de [-0.3, +0.3]. Normalizacao OK.')

## 11. Top/Bottom Tickers por Composite Score

### Sanity check qualitativo

Esta secao serve como **verificacao de sanidade**: os tickers no topo do ranking fazem sentido fundamentalista?  
Se o modelo estiver bem calibrado, devemos ver empresas reconhecidamente solidas no top.

**Exemplos de empresas brasileiras conhecidas por bons fundamentos:**
- **BBAS3** (Banco do Brasil): ROE alto, P/L baixo historicamente, bom DY
- **VALE3** (Vale): ROIC alto em ciclos de commodity, EV/EBIT baixo em momentos de alta de minerio
- **TAEE11** (Taesa): DY elevado consistente, baixa alavancagem, concessao previsivel
- **EGIE3** (Engie): qualidade operacional alta, margem liquida solida, energia renovavel
- **BBSE3** (BB Seguridade): ROE estruturalmente alto, baixo capital de giro, ativo leve

Se empresas problematicas (alto endividamento, prejuizo recorrente) aparecerem no top, revisar os pesos dos scores.

In [ ]:
if 'composite_score' not in df.columns:
    print('composite_score nao encontrado. Pulando secao 11.')
else:
    latest_year = df['year'].max()
    df_latest = df[df['year'] == latest_year].copy().reset_index(drop=True)

    display_cols = ['ticker', 'sectorname', 'label', 'composite_score']
    extra_cols = [c for c in ['value_score', 'quality_score', 'p_l', 'roe', 'dy'] if c in df.columns]
    display_cols += extra_cols

    df_sorted = df_latest[display_cols].sort_values('composite_score', ascending=False)

    top15 = df_sorted.head(15).reset_index(drop=True)
    bot15 = df_sorted.tail(15).reset_index(drop=True)

    fig, axes = plt.subplots(1, 2, figsize=(18, 7))

    # Top 15
    top_colors = [LABEL_COLORS.get(lbl, 'gray') for lbl in top15['label']]
    axes[0].barh(top15['ticker'][::-1], top15['composite_score'][::-1],
                 color=top_colors[::-1], alpha=0.85, edgecolor='white')
    axes[0].set_title(f'Top 15 — Composite Score mais alto ({latest_year})', fontsize=11)
    axes[0].set_xlabel('Composite Score (0-100)')

    # Bottom 15
    bot_colors = [LABEL_COLORS.get(lbl, 'gray') for lbl in bot15['label']]
    axes[1].barh(bot15['ticker'], bot15['composite_score'],
                 color=bot_colors, alpha=0.85, edgecolor='white')
    axes[1].set_title(f'Bottom 15 — Composite Score mais baixo ({latest_year})', fontsize=11)
    axes[1].set_xlabel('Composite Score (0-100)')

    plt.suptitle(f'Ranking de Tickers por Composite Score — Ano {latest_year}', fontsize=12, y=1.01)
    plt.tight_layout()
    plt.show()

    print(f'\nTop 15 tickers por composite_score em {latest_year}:')
    print(top15.round(1).to_string(index=False))

    print(f'\nBottom 15 tickers por composite_score em {latest_year}:')
    print(bot15.round(1).to_string(index=False))

## 12. Importancia de Features — Random Forest como Baseline

### Por que usar Random Forest aqui?

Nesta fase exploratoria, o Random Forest serve como:
1. **Baseline rapido**: antes de otimizar GradientBoosting ou XGBoost, saber o F1 minimo aceitavel
2. **Feature selection**: identificar quais features o modelo considera informativas
3. **Deteccao de leakage**: se uma feature tem importancia suspeitamente alta, pode estar vazando o target

### Anti-leakage — colunas proibidas

As seguintes colunas NAO devem entrar no modelo, pois contem informacao do futuro ou sao derivadas do target:
- `alpha` — e o que foi usado para calcular o label
- `label` — e o proprio target
- `ticker` — identificador, nao feature
- `year` — usar com cuidado (pode capturar ciclos, mas tambem pode causar vazamento temporal)
- Qualquer coluna de retorno futuro

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

FORBIDDEN = {'alpha', 'label', 'ticker', 'year', 'sectorname', 'stock_return',
             'ibov_return', 'next_year_return', 'future_return', 'target'}

feature_cols = [
    c for c in df.columns
    if c not in FORBIDDEN
    and df[c].dtype in [np.float64, np.float32, np.int64, np.int32]
]

print(f'Features candidatas: {len(feature_cols)}')
print(f'Colunas proibidas removidas: {[c for c in FORBIDDEN if c in df.columns]}')

df_ml = df[feature_cols + ['label']].copy()
# Imputar com mediana para o RF nao reclamar de NaN
for col in feature_cols:
    median_val = df_ml[col].median()
    df_ml[col] = df_ml[col].fillna(median_val)
df_ml = df_ml.dropna(subset=['label'])

le = LabelEncoder()
y = le.fit_transform(df_ml['label'])
X = df_ml[feature_cols].values

print(f'\nDataset para RF: {X.shape[0]} amostras x {X.shape[1]} features')
print(f'Classes: {dict(zip(le.classes_, np.bincount(y)))}')

In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import f1_score

rf = RandomForestClassifier(
    n_estimators=200,
    class_weight='balanced',
    max_depth=8,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(rf, X, y, cv=cv, scoring='f1_weighted', n_jobs=-1)

print(f'5-Fold CV F1-weighted: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})')
print(f'Scores por fold: {[f"{s:.4f}" for s in cv_scores]}')

# Treinar para importancia de features
rf.fit(X, y)

importances = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
cum_importance = importances.cumsum()
n_top = min(25, len(importances))

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Top-N importancias
top_imp = importances.head(n_top)
axes[0].barh(top_imp.index[::-1], top_imp.values[::-1],
             color='steelblue', alpha=0.85, edgecolor='white')
axes[0].set_title(f'Top {n_top} Features mais importantes (RF)', fontsize=11)
axes[0].set_xlabel('Importancia (Gini)')

# Importancia cumulativa
axes[1].plot(range(1, len(cum_importance) + 1), cum_importance.values * 100,
             color='steelblue', linewidth=2)
axes[1].axhline(80, color='#f39c12', linestyle='--', linewidth=1.5, label='80%')
axes[1].axhline(90, color='#e74c3c', linestyle='--', linewidth=1.5, label='90%')
axes[1].set_title('Importancia Cumulativa', fontsize=11)
axes[1].set_xlabel('N features')
axes[1].set_ylabel('% importancia acumulada')
axes[1].legend()

plt.suptitle(f'Importancia de Features — Random Forest Baseline (F1-w={cv_scores.mean():.3f})',
             fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

print('\nTop 15 features:')
print(importances.head(15).round(4).to_string())

low_importance = importances[importances < 0.005]
print(f'\nFeatures com importancia < 0.5% ({len(low_importance)} features): possiveis candidatas a remocao')
if len(low_importance) > 0:
    print(low_importance.round(4).to_string())

### Padroes esperados e red flags

**Padroes esperados (sinal de que o pipeline esta correto):**
- Z-scores setoriais aparecem entre as features mais importantes (normalizacao agrega valor)
- `composite_score` ou scores fatoriais estao no top 5 (construcao dos scores e valida)
- CAGR features tem importancia moderada (crescimento e relevante, mas nao dominante)

**Red flags (indicam problema a corrigir antes do Sprint 3):**
- `year` no top 5: o modelo esta aprendendo ciclos de mercado, nao fundamentos — risco de overfitting temporal
- Features de preco (cotacao, volume) no top: leakage potencial — remover imediatamente
- Importancia muito concentrada (1 feature > 50%): modelo fraco, provavelmente um unico indicador dominante

## 13. Analise Temporal — Drift do Composite Score

### Concept drift em dados financeiros

**Concept drift** ocorre quando a relacao entre features e target muda ao longo do tempo.  
Em financas, isso e comum: o que era 'barato' em 2010 pode nao ser barato em 2023 (taxas de juros, inflacao, ciclos setoriais).

### O que analisar

- **Mediana do composite_score por label e ano:** BARATA deve ter score consistentemente maior que CARA
- **Gap BARATA-CARA:** o quanto o score separa as classes
- **Anos com inversao (gap negativo):** periodos onde o score falhou em separar as classes — possivel drift ou evento extremo

**Gap pequeno ou negativo** pode indicar:
1. Ano de crise onde fundamentalismo perdeu para o panico (COVID 2020, Lehman 2008)
2. Ano onde commodities subiram independente de fundamentos (2021-2022)
3. Problema na construcao do label naquele periodo

In [ ]:
if 'composite_score' not in df.columns:
    print('composite_score nao encontrado. Pulando secao 13.')
else:
    years_all = sorted(df['year'].unique())

    # Mediana por label e ano
    score_by_year = df.groupby(['year', 'label'])['composite_score'].median().unstack()

    # Gap BARATA - CARA por ano
    if 'BARATA' in score_by_year.columns and 'CARA' in score_by_year.columns:
        gap = score_by_year['BARATA'] - score_by_year['CARA']
    else:
        gap = pd.Series(dtype=float)

    fig, axes = plt.subplots(2, 1, figsize=(14, 10))

    # Linhas por label
    for lbl in LABEL_ORDER:
        if lbl in score_by_year.columns:
            axes[0].plot(score_by_year.index, score_by_year[lbl],
                         color=LABEL_COLORS[lbl], linewidth=2.5, marker='o', markersize=5, label=lbl)
    axes[0].set_title('Mediana do Composite Score por Label e Ano', fontsize=12)
    axes[0].set_ylabel('Composite Score (0-100)')
    axes[0].legend(title='Label')
    axes[0].set_xticks(years_all)
    axes[0].set_xticklabels(years_all, rotation=45)

    # Gap por ano
    if len(gap) > 0:
        bar_colors_gap = ['#2ecc71' if g >= 0 else '#e74c3c' for g in gap.values]
        axes[1].bar(gap.index, gap.values, color=bar_colors_gap, alpha=0.85, edgecolor='white')
        axes[1].axhline(0, color='black', linewidth=1, linestyle='--')
        axes[1].set_title('Gap BARATA - CARA (verde = score discrimina bem, vermelho = inversao)', fontsize=12)
        axes[1].set_ylabel('Gap (pontos de score)')
        axes[1].set_xticks(gap.index)
        axes[1].set_xticklabels(gap.index, rotation=45)

    plt.suptitle('Analise Temporal — Concept Drift no Composite Score', fontsize=13, y=1.01)
    plt.tight_layout()
    plt.show()

    if len(gap) > 0:
        print(f'Gap medio BARATA-CARA: {gap.mean():.2f} pontos')
        inversoes = gap[gap < 0]
        if len(inversoes) > 0:
            print(f'Anos com inversao (gap negativo): {list(inversoes.index)}')
            print('  -> Nesses anos o score nao discriminou bem as classes. Investigar.')
        else:
            print('Nenhum ano com inversao. Score discrimina bem em todos os periodos.')

## 14. Conclusoes e Proximos Passos

Esta secao consolida os achados do EDA em acoes concretas para o Sprint 3.

In [ ]:
print('=' * 60)
print('RESUMO EXECUTIVO — EDA InvestLink')
print('=' * 60)

print(f'\nDATASET:')
print(f'  Shape:          {df.shape}')
print(f'  Tickers:        {df["ticker"].nunique()}')
print(f'  Periodo:        {df["year"].min()} - {df["year"].max()}')
print(f'  Setores:        {df["sectorname"].nunique()}')

print(f'\nBALANCEAMENTO DE CLASSES:')
for lbl in LABEL_ORDER:
    n_lbl = (df['label'] == lbl).sum()
    pct_lbl = n_lbl / len(df) * 100
    print(f'  {lbl:8s}: {n_lbl:5d} ({pct_lbl:.1f}%)')

print(f'\nQUALIDADE DOS DADOS:')
total_null_pct = df[existing_indicators].isna().mean().mean() * 100
print(f'  % nulos medio nos indicadores: {total_null_pct:.1f}%')
n_criticos = (df[existing_indicators].isna().mean() * 100 > 20).sum()
print(f'  Indicadores criticos (> 20% nulos): {n_criticos}')

if z_cols:
    z_means = df[z_cols].mean()
    z_problemas = (z_means.abs() > 0.3).sum()
    print(f'\nQUALIDADE DOS Z-SCORES:')
    print(f'  Z-scores disponiveis: {len(z_cols)}')
    print(f'  Com media fora de [-0.3, +0.3]: {z_problemas}')

if 'cv_scores' in dir():
    print(f'\nBASELINE RF:')
    print(f'  F1-weighted (5-fold CV): {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})')

print(f'\nPROXIMOS PASSOS:')
print('  1. Executar treinamento completo: python pipeline.py --train')
print('  2. Avaliar modelos:               python pipeline.py --evaluate')
print('  3. Gerar predicoes:               python pipeline.py --predict')
print('  4. Pipeline completo:             python pipeline.py --all')

### 1. Features: Manter / Remover / Testar

| Feature | Decisao | Motivo |
|---------|---------|--------|
| `composite_score` | Manter | Score agregado, alta importancia esperada |
| `*_score` (4 fatores) | Manter | Cada um captura dimensao diferente |
| `*_z` (z-scores) | Manter | Normalizacao setorial essencial |
| `roe`, `roic` | Manter | Alta discriminacao no Kruskal-Wallis |
| `div_liq_ebit` | Manter | Risco de credito bem capturado |
| `dy` | Manter | DY alto correlacionado com BARATA |
| `m_bruta`, `m_ebitda` | Remover | Alta correlacao com `m_liquida` e `m_ebit` |
| `roa` | Remover | Alta correlacao com `roe` e `roic` |
| `div_liq_pl`, `div_liq_ebitda` | Remover | Redundante com `div_liq_ebit` |
| `p_cap_giro` | Testar | Alta % de nulos (bancos ausentes estruturalmente) |
| `peg_ratio` | Testar | Pode ser informativo quando disponivel |
| `graham_formula` | Testar | Util como feature binaria (abaixo/acima) |
| `year` | Nao usar | Risco de aprender ciclos em vez de fundamentos |

---

### 2. Estrategia de Treinamento

- **Sample weights:** `class_weight='balanced'` para compensar desequilibrio de classes
- **Validacao:** `StratifiedKFold(n_splits=5)` para manter proporcao de classes em cada fold
- **Metrica primaria:** `F1-weighted` (penaliza igualmente erros em todas as classes)
- **Metrica secundaria:** `ROC-AUC` (robusta a desequilibrio de classes)
- **Evitar:** acuracia simples como metrica principal (enganosa com classes desbalanceadas)

---

### 3. Hyperparametros Prioritarios

```python
# GradientBoosting (sklearn)
gb_param_grid = {
    'n_estimators':    [100, 200, 300],
    'learning_rate':   [0.01, 0.05, 0.1],
    'max_depth':       [3, 4, 5],
    'min_samples_leaf':[10, 20, 30],
    'subsample':       [0.7, 0.8, 1.0],
    'max_features':    ['sqrt', 'log2']
}

# XGBoost
xgb_param_grid = {
    'n_estimators':    [100, 200, 300],
    'learning_rate':   [0.01, 0.05, 0.1],
    'max_depth':       [3, 4, 6],
    'min_child_weight':[5, 10, 20],
    'subsample':       [0.7, 0.8, 1.0],
    'colsample_bytree':[0.7, 0.8, 1.0],
    'reg_alpha':       [0, 0.1, 1.0],
    'reg_lambda':      [1.0, 5.0, 10.0]
}
```

---

### 4. Validacao do Modelo

- **Walk-forward validation:** treinar em 2008-2018, testar em 2019-2023 (nao misturar passado e futuro)
- **Calibracao de probabilidades:** usar `CalibratedClassifierCV` para que `predict_proba` seja confiavel
- **Threshold tuning:** ajustar limiar de classificacao para maximizar F1 de BARATA especificamente
- **Matriz de confusao:** erro BARATA->CARA e mais grave que BARATA->NEUTRA (false negative de oportunidade)

---

### 5. Melhorias Futuras

- **Dados macroeconomicos:** SELIC, IPCA, cambio BRL/USD como features exogenas
- **Features de momentum:** retorno dos ultimos 3/6/12 meses (alem do alpha anual)
- **Revisao do limiar:** testar +-10% e +-20% no GridSearch para avaliar sensibilidade
- **Ensemble setorial:** modelos especializados por setor (banco, energia, commodities)
- **Deteccao de outliers:** marcar tickers com comportamento atipico (M&A, scandalo, etc)

---

### Proximos passos

Com o EDA concluido, o treinamento e avaliacao completos podem ser executados com:

```bash
# Treinar GradientBoosting e XGBoost com GridSearchCV
python pipeline.py --train --evaluate

# Pipeline completo (scraping + dataset + treino + predicoes)
python pipeline.py --all
```

Os artefatos do modelo serao salvos em `models/artifacts/` (best_model.joblib, scaler.joblib, metadata.json).